# Práctica 6 — K-NN en Dataset MNIST Dígitos
**Machine Learning · ISTER 2026 · Ing. David Minango, PhD**

---
**Objetivo:** Aplicar K-NN al reconocimiento de dígitos manuscritos. Explorar el efecto de K, comparar pesos uniform vs distance, y analizar qué dígitos confunde el modelo.

**Dataset:** load_digits de scikit-learn — 1,797 imágenes 8×8, 10 clases (dígitos 0–9)

⚠️ **Instrucciones:**
- Celdas marcadas con `# 🔧 TU CÓDIGO` debes completarlas.
- Responde las preguntas ❓ en celdas Markdown.
- Guarda una copia en Drive: `Archivo → Guardar una copia en Drive`

## Parte 0 — Setup (Solo ejecutar)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets        import load_digits
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics         import (classification_report, ConfusionMatrixDisplay)

print('✅ Librerías cargadas')

## Parte 1 — Exploración del Dataset

In [ ]:
digits = load_digits()
X = digits.data
y = digits.target

print(f'Imágenes: {digits.images.shape}')
print(f'X shape:  {X.shape}  ({X.shape[1]} features = 8×8 píxeles)')
print(f'Clases:   {digits.target_names}')
print(f'Muestras por clase: {np.bincount(y)}')

In [ ]:
# 🔧 TU CÓDIGO
# Visualiza 1 ejemplo de cada dígito (del 0 al 9) usando imshow
# Usa plt.subplots(2, 5, figsize=(10, 4))
# Para cada dígito d: busca el primer índice donde y == d
# Muestra la imagen y pon el dígito como título

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for d, ax in zip(range(10), axes.flat):
    # ___ tu código aquí ___
    pass

plt.suptitle('Un ejemplo de cada dígito — Dataset Digits 8×8', fontsize=11)
plt.tight_layout()
plt.show()

### ❓ Preguntas Parte 1
*(Responde aquí en Markdown)*

1. ¿Qué dígitos se ven más similares visualmente?
2. ¿Con cuáles esperas que el modelo tenga más errores?

## Parte 2 — K-NN Básico y Efecto de K

In [ ]:
# 🔧 TU CÓDIGO
# 1. Divide X e y: 75% train, 25% test, random_state=42, stratify=y
# 2. Normaliza con StandardScaler (fit en train, transform en ambos)
# 3. Entrena KNeighborsClassifier(n_neighbors=5) con datos normalizados
# 4. Imprime test accuracy y classification report

X_train, X_test, y_train, y_test = ___________

scaler  = StandardScaler()
X_tr_sc = ___________
X_te_sc = ___________

knn5 = ___________
knn5.fit(___________, ___________)

acc_test = knn5.score(X_te_sc, y_test)
print(f'K-NN (k=5) — Test accuracy: {acc_test:.4f}')
print(classification_report(y_test, knn5.predict(X_te_sc)))

In [ ]:
# 🔧 TU CÓDIGO
# Entrena K-NN con K de 1 a 20 usando weights='uniform'
# Para cada K: calcula CV-5 accuracy sobre (X_tr_sc, y_train)
# Grafica K vs CV-5 accuracy con línea vertical en el mejor K

k_vals  = range(1, 21)
cv_accs = []

for k in k_vals:
    knn  = KNeighborsClassifier(n_neighbors=k, weights='uniform')
    score = cross_val_score(knn, X_tr_sc, y_train, cv=5, scoring='accuracy').mean()
    cv_accs.append(score)

mejor_k = list(k_vals)[cv_accs.index(max(cv_accs))]
print(f'Mejor K (uniform): {mejor_k}  | CV-5 acc: {max(cv_accs):.4f}')

plt.figure(figsize=(9, 4))
# ___ graficar curva y línea vertical ___
plt.xlabel('K (número de vecinos)')
plt.ylabel('CV-5 Accuracy')
plt.title('Curva de K óptimo — K-NN en Digits (weights=uniform)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### ❓ Preguntas Parte 2
*(Responde aquí en Markdown)*

1. ¿K=1 tiene alta o baja accuracy en CV? ¿Qué dice eso sobre los datos?
2. ¿A partir de qué K empieza a bajar la accuracy?

## Parte 3 — Pesos Uniform vs Distance

In [ ]:
# 🔧 TU CÓDIGO
# Repite la curva de K (1 a 20) con weights='distance'
# Grafica ambas curvas en el mismo plot

cv_distance = []
for k in k_vals:
    knn   = KNeighborsClassifier(n_neighbors=k, weights='distance')
    score = cross_val_score(knn, X_tr_sc, y_train, cv=5, scoring='accuracy').mean()
    cv_distance.append(score)

mejor_k_dist = list(k_vals)[cv_distance.index(max(cv_distance))]
print(f'Mejor K (distance): {mejor_k_dist}  | CV-5 acc: {max(cv_distance):.4f}')

plt.figure(figsize=(9, 4))
plt.plot(list(k_vals), cv_accs,    'o-', color='#3b82f6', lw=2, label='weights=uniform')
plt.plot(list(k_vals), cv_distance, 's-', color='#10b981', lw=2, label='weights=distance')
plt.axvline(mejor_k,      color='#3b82f6', linestyle='--', lw=1.2)
plt.axvline(mejor_k_dist, color='#10b981', linestyle='--', lw=1.2)
plt.xlabel('K')
plt.ylabel('CV-5 Accuracy')
plt.title('Comparativa pesos: uniform vs distance — Digits')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### ❓ Preguntas Parte 3
*(Responde aquí en Markdown)*

1. ¿Qué combinación (K + weights) da la mayor accuracy?
2. ¿Por qué weights='distance' puede ser mejor en Digits?

## Parte 4 — Análisis de Errores

In [ ]:
# 🔧 TU CÓDIGO
# 1. Entrena el mejor K-NN (mejor K y mejor weights)
# 2. Predice sobre X_te_sc
# 3. Muestra ConfusionMatrixDisplay

knn_best = KNeighborsClassifier(n_neighbors=mejor_k_dist, weights='distance')
knn_best.fit(X_tr_sc, y_train)
y_pred = knn_best.predict(X_te_sc)

print(f'Test accuracy del mejor K-NN: {knn_best.score(X_te_sc, y_test):.4f}')

fig, ax = plt.subplots(figsize=(8, 7))
# ___ ConfusionMatrixDisplay ___
ax.set_title('Matriz de Confusión — K-NN en Digits', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# 🔧 TU CÓDIGO
# Encuentra los índices donde y_pred != y_test
# Visualiza los primeros 15 errores en un grid 3×5
# Para cada error: imagen + título 'Real: X | Pred: Y'

errores = np.where(y_pred != y_test)[0]
print(f'Total de errores: {len(errores)} de {len(y_test)} ({len(errores)/len(y_test)*100:.1f}%)')

fig, axes = plt.subplots(3, 5, figsize=(12, 7))
# Para reconstruir la imagen desde X_te_sc necesitamos desnormalizar o usar X_test directamente
X_test_orig = X_test  # guardado antes de escalar
for ax, idx in zip(axes.flat, errores[:15]):
    # ___ tu código ___
    pass

plt.suptitle('Errores del K-NN en Digits — Real vs Predicho', fontsize=11)
plt.tight_layout()
plt.show()

### ❓ Preguntas Parte 4
*(Responde aquí en Markdown)*

1. ¿Qué par de dígitos aparece más confundido en la matriz?
2. ¿Los errores fueron 'razonables' al ver las imágenes?
3. ¿Cómo mejorarías K-NN en este dataset?

## 🌟 Desafío Bonus — K-NN con PCA (Opcional)

In [ ]:
# 🔧 DESAFÍO
# Aplica PCA para reducir de 64 a {10, 20, 30, 40} componentes antes de K-NN
# Para cada número de componentes: CV-5 accuracy con el mejor K-NN
# ¿PCA mejora K-NN?

from sklearn.decomposition import PCA
from sklearn.pipeline      import Pipeline

for n_comp in [10, 20, 30, 40]:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('pca',    PCA(n_components=n_comp)),
        ('knn',    KNeighborsClassifier(n_neighbors=mejor_k_dist, weights='distance'))
    ])
    score = cross_val_score(pipe, X, y, cv=5, scoring='accuracy').mean()
    print(f'PCA {n_comp:2d} componentes | CV-5 acc: {score:.4f}')

## 📝 Conclusiones

*(Escribe aquí tu párrafo de conclusiones)*

Responde: ¿Cuál fue el K óptimo y los pesos elegidos? ¿Qué dígitos confunde más el modelo y por qué? ¿Qué harías para mejorar el rendimiento del K-NN en este dataset?